In [ ]:
# Python ≥3.8 is required
import sys
assert sys.version_info >= (3, 8)

# Scikit-Learn ≥ 1.0 is required
import sklearn
assert sklearn.__version__ >= "1.0"

# Common imports
import numpy as np
import pandas as pd
import os

# To plot figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Precision options
np.set_printoptions(precision=2)
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Python ≥3.9 is required
import sys
assert sys.version_info >= (3, 9)

import seaborn as sns

# Statistical analysis and testing
from scipy import stats
from statsmodels.formula.api import ols
import statsmodels.api as sm

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

In [ ]:
train_import = pd.read_csv("train_bin_nhanes.csv")
test_import = pd.read_csv("test_bin_nhanes.csv")

In [ ]:
columns_to_remove = [
    "LBXGLU",
    "URDACT",
    "RIDAGEYR",
    "BMXBMI",
    "LBDHDD",
    "INDFMPIR",
    "LBXIN",
    "SLD012",
    "LBXTLG",
    "SLD013"
]

train_import = train_import.drop(columns=columns_to_remove)
test_import = test_import.drop(columns=columns_to_remove)

In [ ]:
X_train = train_import.drop(columns=["target"])
y_train = train_import["target"]

X_test = test_import.drop(columns=["target"])
y_test = test_import["target"]

In [ ]:
# Display scores function
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", f"{scores.mean():.2f}")
    print("Standard deviation:", f"{scores.std():.2f}")

In [ ]:
#reduced to 5 to reduce time
from sklearn.model_selection import cross_validate, KFold
n_splits = 5
k_fold = KFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(y_train.unique())

(5372, 56)
(5372,)
[0 1]


### Model 1 - Random Forest

In [ ]:
#extreme overfitting
from sklearn.ensemble import RandomForestClassifier

rf_extreme = RandomForestClassifier(
    n_estimators=100,
    max_features="sqrt",
    min_samples_split=2,
    min_samples_leaf=1,
    max_depth=None,
    bootstrap=True,
    random_state=77
)

rf_extreme.fit(X_train, y_train)


In [ ]:
#extreme overfitting training and test performance
from sklearn.metrics import accuracy_score, roc_auc_score

# training performance
y_train_pred_extreme = rf_extreme.predict(X_train)
y_train_probs_extreme = rf_extreme.predict_proba(X_train)[:, 1]

train_accuracy_extreme = accuracy_score(
    y_train,
    y_train_pred_extreme
)

train_auc_extreme = roc_auc_score(
    y_train,
    y_train_probs_extreme
)

# hold-out/test performance
y_test_pred_extreme = rf_extreme.predict(X_test)
y_test_probs_extreme = rf_extreme.predict_proba(X_test)[:, 1]

test_accuracy_extreme = accuracy_score(
    y_test,
    y_test_pred_extreme
)

test_auc_extreme = roc_auc_score(
    y_test,
    y_test_probs_extreme
)

print("Extreme Overfitting")
print("Training ROC-AUC:", train_auc_extreme)
print("Hold-out ROC-AUC:", test_auc_extreme)


In [ ]:
import numpy as np
from sklearn.utils import resample
from sklearn.metrics import roc_auc_score

def bootstrap_optimism_auc(model, X_train, y_train, B=50):

    optimisms = []

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    for i in range(B):

        # bootstrap sample from training data
        X_boot, y_boot = resample(
            X_train,
            y_train,
            replace=True,
            random_state=i
        )

        # fit model to bootstrap sample
        model.fit(X_boot, y_boot)

        # apparent AUC on bootstrap sample
        boot_probs = model.predict_proba(X_boot)[:, 1]

        boot_auc = roc_auc_score(
            y_boot,
            boot_probs
        )

        # AUC on original training data
        original_train_probs = model.predict_proba(X_train)[:, 1]

        original_train_auc = roc_auc_score(
            y_train,
            original_train_probs
        )

        # optimism
        optimism = boot_auc - original_train_auc

        optimisms.append(optimism)

    # average optimism
    avg_optimism = np.mean(optimisms)

    # fit final model on full training data
    model.fit(X_train, y_train)

    # apparent AUC on full training data
    apparent_final = roc_auc_score(
        y_train,
        model.predict_proba(X_train)[:, 1]
    )

    # optimism-corrected AUC
    corrected = apparent_final - avg_optimism

    return {
        "apparent": apparent_final,
        "optimism": avg_optimism,
        "corrected": corrected
    }


rf_extreme_bootstrap = bootstrap_optimism_auc(
    rf_extreme,
    X_train,
    y_train,
    B=50
)

print("Extreme Overfitting - Bootstrap")
print("Apparent AUC:", rf_extreme_bootstrap["apparent"])
print("Optimism:", rf_extreme_bootstrap["optimism"])
print("Corrected AUC:", rf_extreme_bootstrap["corrected"])


In [ ]:
rf_moderate = RandomForestClassifier(
    n_estimators=100,
    max_features=1,
    max_depth=2,
    min_samples_split=800,
    min_samples_leaf=300,
    max_leaf_nodes=3,
    bootstrap=True,
    max_samples=0.25,
    random_state=77
)

rf_moderate.fit(X_train, y_train)

In [ ]:
# training performance
y_train_pred_moderate = rf_moderate.predict(X_train)
y_train_probs_moderate = rf_moderate.predict_proba(X_train)[:, 1]

train_accuracy_moderate = accuracy_score(
    y_train,
    y_train_pred_moderate
)

train_auc_moderate = roc_auc_score(
    y_train,
    y_train_probs_moderate
)

# hold-out/test performance
y_test_pred_moderate = rf_moderate.predict(X_test)
y_test_probs_moderate = rf_moderate.predict_proba(X_test)[:, 1]

test_accuracy_moderate = accuracy_score(
    y_test,
    y_test_pred_moderate
)

test_auc_moderate = roc_auc_score(
    y_test,
    y_test_probs_moderate
)

print("No Overfitting")
print("Training ROC-AUC:", train_auc_moderate)
print("Hold-out ROC-AUC:", test_auc_moderate)


In [ ]:
rf_moderate_bootstrap = bootstrap_optimism_auc(
    rf_moderate,
    X_train,
    y_train,
    B=50
)

print("No Overfitting - Bootstrap")
print("Apparent AUC:", rf_moderate_bootstrap["apparent"])
print("Optimism:", rf_moderate_bootstrap["optimism"])
print("Corrected AUC:", rf_moderate_bootstrap["corrected"])